# ChatGPT Archive Compiler — clean Colab bootstrap

This notebook resolves a repository branch to an exact Git commit, creates or reuses a commit-specific checkout under **MyDrive/ChatGPT Data Export/checkouts**, installs the package from that checkout, validates single- and multipart ingestion with synthetic data, and optionally ingests a real export.

The checkout workflow is intentionally immutable: it never pulls, switches, merges, stashes, resets, deletes, or overwrites an existing repository. A clean checkout at the exact commit is reused; an incompatible or dirty path is left untouched and a new commit-specific retry directory is created.

Before running, add a Colab secret named `GITHUB_TOKEN` containing a fine-grained GitHub token with read-only Contents access to the private `jcollins-bioinfo/chatgpt-archive-compiler` repository, and enable **Notebook access** for that secret. The token is supplied through a temporary `GIT_ASKPASS` helper and never appears in a URL, command argument, Git configuration, or notebook output.

Privacy boundary: Colab runs on Google-hosted infrastructure. Synthetic validation is enabled by default. Real-export ingestion is disabled until you provide an exact ZIP path and explicit acknowledgment. No cell prints conversation titles or message content.


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

In [ ]:
from pathlib import Path

REPO_BRANCH = "agent/rebuild-colab-workflow"  # @param {type:"string"}
REPO_FULL_NAME = "jcollins-bioinfo/chatgpt-archive-compiler"
PUBLIC_REPO_URL = f"https://github.com/{REPO_FULL_NAME}.git"
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/ChatGPT Data Export")
CHECKOUT_ROOT = DRIVE_PROJECT_DIR / "checkouts" / "chatgpt-archive-compiler"
OUTPUT_ROOT = DRIVE_PROJECT_DIR / "outputs"

if not DRIVE_PROJECT_DIR.is_dir():
    raise RuntimeError("Expected Drive folder is missing: MyDrive/ChatGPT Data Export")
CHECKOUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Drive project directory: {DRIVE_PROJECT_DIR}")
print(f"Repository branch: {REPO_BRANCH}")

In [ ]:
import os
import re
import subprocess
import sys
import tempfile
from collections.abc import Iterator, Mapping, Sequence
from contextlib import contextmanager
from datetime import UTC, datetime
from urllib.parse import urlsplit


def run_command(
    command: Sequence[str],
    *,
    cwd: Path | None = None,
    env: Mapping[str, str] | None = None,
    capture_output: bool = False,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    """Run one shell-free subprocess with explicit arguments."""

    return subprocess.run(
        list(command),
        cwd=cwd,
        env=dict(env) if env is not None else None,
        check=check,
        text=True,
        capture_output=capture_output,
    )


def get_github_token() -> str:
    """Read the private-repository token without exposing provider error text."""

    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError(
            "Colab secret GITHUB_TOKEN is missing or Notebook access is disabled."
        ) from None
    if not token:
        raise RuntimeError("Colab secret GITHUB_TOKEN is empty.")
    return token


ASKPASS_SOURCE = """#!/usr/bin/env python3
import os
import sys

prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ""
print("x-access-token" if "username" in prompt else os.environ["CAC_GIT_TOKEN"])
"""


@contextmanager
def authenticated_git_environment() -> Iterator[dict[str, str]]:
    """Yield a subprocess environment backed by a temporary askpass helper."""

    token = get_github_token()
    with tempfile.TemporaryDirectory(prefix="cac_git_auth_") as temporary_directory:
        helper = Path(temporary_directory) / "askpass.py"
        helper.write_text(ASKPASS_SOURCE, encoding="utf-8")
        helper.chmod(0o700)
        environment = os.environ.copy()
        environment.pop("GITHUB_TOKEN", None)
        environment.update(
            {
                "CAC_GIT_TOKEN": token,
                "GIT_ASKPASS": str(helper),
                "GIT_TERMINAL_PROMPT": "0",
            }
        )
        try:
            yield environment
        finally:
            environment.pop("CAC_GIT_TOKEN", None)


def origin_targets_expected_repository(origin: str) -> bool:
    """Return whether an origin URL names the configured GitHub repository."""

    if origin.startswith("git@github.com:"):
        repository_path = origin.split(":", 1)[1]
    else:
        parsed = urlsplit(origin)
        if parsed.hostname != "github.com":
            return False
        repository_path = parsed.path
    return repository_path.strip("/").removesuffix(".git") == REPO_FULL_NAME


def resolve_remote_commit() -> str:
    """Resolve the selected branch to one exact 40-character Git commit SHA."""

    valid_branch = run_command(
        ["git", "check-ref-format", "--branch", REPO_BRANCH],
        capture_output=True,
        check=False,
    ).returncode
    if valid_branch != 0:
        raise ValueError("REPO_BRANCH is not a valid Git branch name.")

    expected_ref = f"refs/heads/{REPO_BRANCH}"
    with authenticated_git_environment() as environment:
        result = run_command(
            [
                "git",
                "-c",
                "credential.helper=",
                "ls-remote",
                "--exit-code",
                PUBLIC_REPO_URL,
                expected_ref,
            ],
            env=environment,
            capture_output=True,
        )
    matches = [
        fields[0]
        for line in result.stdout.splitlines()
        if len(fields := line.split()) == 2 and fields[1] == expected_ref
    ]
    if len(matches) != 1 or re.fullmatch(r"[0-9a-f]{40}", matches[0]) is None:
        raise RuntimeError("Selected branch did not resolve to exactly one Git commit.")
    return matches[0]


def checkout_is_reusable(path: Path, commit_sha: str) -> bool:
    """Return whether a checkout is clean, exact, and points at the expected origin."""

    if not (path / ".git").is_dir():
        return False
    origin_result = run_command(
        ["git", "remote", "get-url", "origin"],
        cwd=path,
        capture_output=True,
        check=False,
    )
    if origin_result.returncode != 0 or not origin_targets_expected_repository(
        origin_result.stdout.strip()
    ):
        return False
    head_result = run_command(
        ["git", "rev-parse", "HEAD"],
        cwd=path,
        capture_output=True,
        check=False,
    )
    if head_result.returncode != 0 or head_result.stdout.strip() != commit_sha:
        return False
    status_result = run_command(
        ["git", "status", "--porcelain", "--untracked-files=all"],
        cwd=path,
        capture_output=True,
        check=False,
    )
    return status_result.returncode == 0 and not status_result.stdout


def prepare_commit_checkout() -> tuple[Path, str, bool]:
    """Reuse an exact checkout or clone the selected commit into a new Drive path."""

    commit_sha = resolve_remote_commit()
    preferred_path = CHECKOUT_ROOT / commit_sha[:12]
    if checkout_is_reusable(preferred_path, commit_sha):
        return preferred_path, commit_sha, True

    target_path = preferred_path
    if target_path.exists():
        suffix = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
        target_path = CHECKOUT_ROOT / f"{commit_sha[:12]}-retry-{suffix}-{os.getpid()}"
    if target_path.exists():
        raise RuntimeError("Fresh commit-specific checkout path unexpectedly exists.")

    with authenticated_git_environment() as environment:
        run_command(
            [
                "git",
                "-c",
                "credential.helper=",
                "clone",
                "--branch",
                REPO_BRANCH,
                "--single-branch",
                "--no-tags",
                PUBLIC_REPO_URL,
                str(target_path),
            ],
            env=environment,
        )
    run_command(["git", "checkout", "--detach", commit_sha], cwd=target_path)
    run_command(
        ["git", "config", "--local", "core.fileMode", "false"],
        cwd=target_path,
    )
    if not checkout_is_reusable(target_path, commit_sha):
        raise RuntimeError("New checkout failed exact-commit or cleanliness verification.")
    return target_path, commit_sha, False

In [ ]:
REPO_DIR, CHECKED_OUT_COMMIT, REUSED_CHECKOUT = prepare_commit_checkout()

print(f"Checkout: {REPO_DIR}")
print(f"Commit: {CHECKED_OUT_COMMIT[:12]}")
print(f"Reused existing exact checkout: {REUSED_CHECKOUT}")

In [ ]:
import importlib

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "-e",
        f"{REPO_DIR}[notebooks]",
    ]
)
repository_source = (REPO_DIR / "src").resolve()
if not repository_source.is_dir():
    raise RuntimeError("Exact checkout does not contain the expected src directory.")
sys.path.insert(0, str(repository_source))
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "chatgpt_archive_compiler" or module_name.startswith(
        "chatgpt_archive_compiler."
    ):
        del sys.modules[module_name]

archive_compiler = importlib.import_module("chatgpt_archive_compiler")
ingest_module = importlib.import_module("chatgpt_archive_compiler.ingest")
serialization_module = importlib.import_module("chatgpt_archive_compiler.serialization")
ArchiveIngestError = ingest_module.ArchiveIngestError
IngestLimits = ingest_module.IngestLimits
SchemaMode = ingest_module.SchemaMode
ingest_export_zip = ingest_module.ingest_export_zip
inspect_zip = ingest_module.inspect_zip
summarize_archive = ingest_module.summarize_archive
read_archive_ir = serialization_module.read_archive_ir
write_archive_ir = serialization_module.write_archive_ir

imported_from = Path(archive_compiler.__file__).resolve()
if repository_source not in imported_from.parents:
    raise RuntimeError("Package import did not resolve to the exact Drive checkout.")
if not checkout_is_reusable(REPO_DIR, CHECKED_OUT_COMMIT):
    raise RuntimeError("Package installation unexpectedly changed the Git checkout.")

print(f"Package version: {archive_compiler.__version__}")
print(f"Imported from: {imported_from}")

## Synthetic multipart validation

This test writes three numbered payload members in reverse ZIP order, then verifies that ingestion processes them in numeric order, preserves every branch, and round-trips the resulting Archive IR. All temporary files remain in the Colab runtime and contain synthetic content only.


In [ ]:
import copy
import json
import zipfile

synthetic_conversation = {
    "id": "synthetic-conversation",
    "title": "Synthetic branched conversation",
    "create_time": 1_735_689_600,
    "update_time": 1_735_689_700,
    "current_node": "assistant-current",
    "mapping": {
        "root": {
            "id": "root",
            "parent": None,
            "children": ["user"],
            "message": None,
        },
        "user": {
            "id": "user",
            "parent": "root",
            "children": ["assistant-current", "assistant-alternate"],
            "message": {
                "id": "message-user",
                "author": {"role": "user"},
                "create_time": 1_735_689_600,
                "content": {"content_type": "text", "parts": ["Question"]},
                "metadata": {},
            },
        },
        "assistant-current": {
            "id": "assistant-current",
            "parent": "user",
            "children": [],
            "message": {
                "id": "message-assistant-current",
                "author": {"role": "assistant"},
                "create_time": 1_735_689_700,
                "content": {"content_type": "text", "parts": ["Current"]},
                "metadata": {},
            },
        },
        "assistant-alternate": {
            "id": "assistant-alternate",
            "parent": "user",
            "children": [],
            "message": {
                "id": "message-assistant-alternate",
                "author": {"role": "assistant"},
                "create_time": 1_735_689_650,
                "content": {"content_type": "text", "parts": ["Alternate"]},
                "metadata": {},
            },
        },
    },
}

with tempfile.TemporaryDirectory(prefix="cac_synthetic_") as temporary_directory:
    temporary_path = Path(temporary_directory)
    fixture_zip = temporary_path / "multipart-export.zip"
    with zipfile.ZipFile(fixture_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive_zip:
        for index in reversed(range(3)):
            conversation = copy.deepcopy(synthetic_conversation)
            conversation["id"] = f"synthetic-conversation-{index}"
            archive_zip.writestr(
                f"conversations-{index:03d}.json",
                json.dumps([conversation], allow_nan=False),
            )
    synthetic_archive = ingest_export_zip(
        fixture_zip,
        limits=IngestLimits(),
        schema_mode=SchemaMode.STRICT,
    )
    synthetic_output = temporary_path / "archive.ir.json"
    write_archive_ir(synthetic_archive, synthetic_output)
    assert read_archive_ir(synthetic_output) == synthetic_archive

synthetic_summary = summarize_archive(synthetic_archive)
assert [conversation.conversation_id for conversation in synthetic_archive.conversations] == [
    f"synthetic-conversation-{index}" for index in range(3)
]
assert [str(path) for path in synthetic_archive.source_manifest.selected_conversation_paths] == [
    f"conversations-{index:03d}.json" for index in range(3)
]
assert synthetic_summary.conversation_count == 3
assert synthetic_summary.node_count == 12
assert synthetic_summary.message_count == 9
assert synthetic_summary.current_path_message_count == 6
assert synthetic_summary.warning_count == 0

print("Synthetic multipart validation passed.")
print(f"Conversations: {synthetic_summary.conversation_count}")
print(f"Payload members: {synthetic_archive.metadata['conversation_payload_count']}")
print(f"Warnings: {synthetic_summary.warning_count}")

## Optional real-export ingestion — disabled by default

Enabling this cell reads the exact ZIP path you provide from Drive and writes a normalized Archive IR beneath `MyDrive/ChatGPT Data Export/outputs/real`. It prints only file counts, aggregate archive counts, warning counts, and the destination path. It does not search Drive or print source filenames, titles, messages, warning contexts, or exception text.


In [ ]:
RUN_REAL_EXPORT = False  # @param {type:"boolean"}
REAL_EXPORT_ZIP = ""  # @param {type:"string"}
PRIVACY_ACKNOWLEDGEMENT = ""  # @param {type:"string"}
REQUIRED_ACKNOWLEDGEMENT = "I UNDERSTAND THIS RUNS IN GOOGLE COLAB"

if not RUN_REAL_EXPORT:
    print("Real-export ingestion remains disabled.")
else:
    if PRIVACY_ACKNOWLEDGEMENT != REQUIRED_ACKNOWLEDGEMENT:
        raise RuntimeError("Exact privacy acknowledgment is required.")
    real_export_path = Path(REAL_EXPORT_ZIP).expanduser()
    if not real_export_path.is_file() or real_export_path.suffix.casefold() != ".zip":
        raise RuntimeError("REAL_EXPORT_ZIP must be an exact existing .zip file path.")

    try:
        source_manifest = inspect_zip(
            real_export_path,
            limits=IngestLimits(),
            compute_hashes=False,
            compute_archive_hash=False,
        )
        candidate_files = [
            source_file
            for source_file in source_manifest.files
            if source_file.detected_kind.value
            in {"conversations_json", "conversations_json_candidate"}
        ]
        print(f"Safe ZIP members: {len(source_manifest.files)}")
        print(f"Conversation payload candidates: {len(candidate_files)}")
        print(f"Candidate JSON bytes: {sum(item.size_bytes for item in candidate_files):,}")
        real_archive = ingest_export_zip(
            real_export_path,
            limits=IngestLimits(),
            schema_mode=SchemaMode.TOLERANT,
        )
    except Exception as exception:
        error_name = type(exception).__name__
        raise RuntimeError(
            f"Real-export ingestion failed safely ({error_name}); source-derived exception "
            "text was suppressed."
        ) from None

    run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    real_output_directory = OUTPUT_ROOT / "real" / f"{run_id}-{CHECKED_OUT_COMMIT[:12]}"
    real_output_directory.mkdir(parents=True, exist_ok=False)
    real_output_path = write_archive_ir(real_archive, real_output_directory / "archive.ir.json")
    real_summary = summarize_archive(real_archive)

    print("Real-export ingestion completed.")
    print(f"Payload members: {real_archive.metadata['conversation_payload_count']}")
    print(f"Conversations: {real_summary.conversation_count}")
    print(f"Graph nodes: {real_summary.node_count}")
    print(f"Messages: {real_summary.message_count}")
    print(f"Current-path messages: {real_summary.current_path_message_count}")
    print(f"Warnings: {real_summary.warning_count}")
    print(f"Archive IR: {real_output_path}")

## Checkout retention

This notebook never removes older Drive checkouts. After a successful run, you may review and manually archive or delete obsolete directories under `MyDrive/ChatGPT Data Export/checkouts/chatgpt-archive-compiler`. Do not apply stashes from earlier checkout attempts to these commit-specific directories.
